# Сравнение стратегий обработки праймеров лёгких цепей мыши

Сравниваются выбранная ветка fastp по минимальной длине с обработкой `paper4-cut` и отдельная ветка `paper4 + эмпирический кандидат Cκ --mode cut`. Мотив Cκ не представлен как опубликованный праймер набора. Риды без совпадения с праймером сохраняются в пайплайне. QC выполняется отдельно в `qc.ipynb`.


In [ ]:
from pathlib import Path
import csv,gzip,json,os,shutil,statistics,subprocess,sys,time
from collections import Counter
START=Path.cwd().resolve();REPO=next((p for p in (START,*START.parents) if (p/'scripts/mixed_chain_truth.py').is_file()),None)
if REPO is None:raise RuntimeError(f'cannot locate repo from {START}')
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from scripts.mixed_chain_truth import build_airr_truth
from scripts.presto_qc import count_fastq_records,parse_presto_summary
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'results/PRJNA1226555').is_dir():VOLUME=REPO
ENV=Path(os.environ.get('BCR_ENV','/Users/epishkin/mamba/envs/bcr_env'))
BASE=VOLUME/'results/PRJNA1226555';GERMLINE=BASE/'references/germline';RUN='SRR32426580'
MINLEN_REPORT=BASE/'comparisons/light_fastp_minlen_comparison.json'
report=json.loads(MINLEN_REPORT.read_text())['comparison']
selected_label=max(('min200','min250'),key=lambda x:report[x]['strict']['accepted_source_records'])
SOURCE_BRANCH=f'fastp_q30_u40_{selected_label}';TARGET_BRANCH=f'{SOURCE_BRANCH}_paper4_plus_empirical_ck_cut'
SOURCE=BASE/'branches'/SOURCE_BRANCH;TARGET_FINAL=BASE/'branches'/TARGET_BRANCH;TARGET=BASE/'branches'/f'{TARGET_BRANCH}.staging'
PRIMER_FASTA=REPO/'seq_refs/mouse_light_paper4_plus_empirical_ck.fasta'
for x in [PRIMER_FASTA,ENV/'bin/MaskPrimers.py',ENV/'bin/PairSeq.py',ENV/'bin/AssemblePairs.py',ENV/'bin/CollapseSeq.py',ENV/'bin/igblastn']:assert x.exists(),x
print({'selected_length':selected_label,'source':str(SOURCE),'target':str(TARGET)})
def run(cmd,out,err,heartbeat=30,env=None):
 out=Path(out);err=Path(err);out.parent.mkdir(parents=True,exist_ok=True);t=time.monotonic()
 with out.open('w') as oh,err.open('w') as eh:
  proc=subprocess.Popen([str(x) for x in cmd],stdout=oh,stderr=eh,text=True,env=env);print('PID',proc.pid,flush=True)
  while proc.poll() is None:print(f'PID={proc.pid} elapsed={(time.monotonic()-t)/60:.1f}min',flush=True);time.sleep(30)
 if proc.returncode:raise RuntimeError(f'rc={proc.returncode}; inspect {err}')
def fasta_count(path):return sum(x.startswith('>') for x in open(path))
def tsv_count(path):return max(0,sum(1 for _ in open(path,errors='replace'))-1)
def fasta_ids(path):
 ids=[]
 with Path(path).open() as h:
  for line in h:
   if line.startswith('>'):ids.append(line[1:].strip().split()[0])
 return ids
def airr_ids(path):
 with Path(path).open(errors='replace') as h:return [r['sequence_id'] for r in csv.DictReader(h,delimiter='\t')]
def assert_airr_query_identity(fasta,airr):
 expected=fasta_ids(fasta);observed=airr_ids(airr)
 if len(expected)!=len(set(expected)):raise AssertionError(f'duplicate FASTA IDs: {len(expected)-len(set(expected))}')
 if len(observed)!=len(set(observed)):raise AssertionError(f'duplicate AIRR sequence_ids: {len(observed)-len(set(observed))}')
 missing=set(expected)-set(observed);extra=set(observed)-set(expected)
 assert not missing and not extra,(f'AIRR query-ID mismatch: missing={len(missing)} extra={len(extra)}',sorted(missing)[:5],sorted(extra)[:5])
 assert len(observed)==len(expected),(len(observed),len(expected))
 return len(expected)


In [ ]:
# Удаление праймеров и синхронизация пар; выбранные trimmed FASTQ используются только для чтения.
t1=SOURCE/'trimmed/fastq'/f'{RUN}_1.trim.fastq.gz';t2=SOURCE/'trimmed/fastq'/f'{RUN}_2.trim.fastq.gz'
for x in (t1,t2):assert x.exists(),x
shutil.rmtree(TARGET,ignore_errors=True)
PR=TARGET/'pr_trimmed';MASK=PR/'mask';SYNC=PR/'sync';LOG=PR/'logs'
for d in (MASK,SYNC,LOG):d.mkdir(parents=True)
(PR/'source_branch.json').write_text(json.dumps({'source_branch':SOURCE_BRANCH,'primer_strategy':'paper4_plus_empirical_ck_cut','ck_source_status':'empirical_candidate_not_kit_source_backed','mode':'cut'},indent=2)+chr(10))
run([ENV/'bin/MaskPrimers.py','align','-s',t1,'-p',PRIMER_FASTA,'--mode','cut','--maxerror','0.2','--maxlen','50','--nproc','4','--failed','--gzip-output','--outdir',MASK,'--outname',f'{RUN}_R1_LIGHT'],LOG/'mask.stdout.log',LOG/'mask.stderr.log')
pass_r1=next(MASK.glob('*primers-pass.fastq.gz'));fail_r1=next(MASK.glob('*primers-fail.fastq.gz'))
assert count_fastq_records(pass_r1)+count_fastq_records(fail_r1)==count_fastq_records(t1)
def sync(name,subset):
 out=SYNC/name;out.mkdir()
 run([ENV/'bin/PairSeq.py','-1',subset,'-2',t2,'--coord','sra','--gzip-output','--outdir',out,'--outname',f'{RUN}_{name}'],LOG/f'pair_{name}.stdout.log',LOG/f'pair_{name}.stderr.log')
 files=sorted(out.glob('*pair-pass*.fastq.gz'));assert len(files)==2,files
 r1=next(x for x in files if '-1_pair-pass' in x.name);r2=next(x for x in files if '-2_pair-pass' in x.name)
 assert count_fastq_records(r1)==count_fastq_records(r2)==count_fastq_records(subset)
 return r1,r2
pass_pair=sync('primer_pass',pass_r1);unmatched_pair=sync('primer_unmatched',fail_r1)
summary={'trimmed_pairs':count_fastq_records(t1),'primer_pass_pairs':count_fastq_records(pass_pair[0]),'primer_unmatched_pairs':count_fastq_records(unmatched_pair[0])}
(PR/'primer_summary.json').write_text(json.dumps(summary,indent=2)+chr(10));print(summary)


In [ ]:
# Объединение pRESTO для обеих сохраняемых веток: R2 — head, R1 — tail.
MERGED=TARGET/'merged';MLOG=MERGED/'logs';shutil.rmtree(MERGED,ignore_errors=True);MLOG.mkdir(parents=True)
def assemble(name,pair,copy_primer=False):
 r1,r2=pair;out=MERGED/name;out.mkdir();stdout=MLOG/f'{name}.stdout.log';stderr=MLOG/f'{name}.stderr.log'
 cmd=[ENV/'bin/AssemblePairs.py','align','-1',r2,'-2',r1,'--coord','sra','--rc','tail','--nproc','4','--failed','--gzip-output','--outdir',out,'--outname',f'{RUN}_{name}']
 if copy_primer:cmd+=['--2f','PRIMER']
 run(cmd,stdout,stderr)
 ps=list(out.glob('*assemble-pass.fastq.gz'));fs=list(out.glob('*assemble-fail.fastq.gz'));assert len(ps)==1 and len(fs)==2
 presto=parse_presto_summary(stdout.read_text(errors='replace'));assert count_fastq_records(ps[0])==presto['pass'];assert count_fastq_records(fs[0])==count_fastq_records(fs[1])==presto['fail']
 return ps[0],presto
primer_pass,primer_presto=assemble('primer_pass',pass_pair,True)
unmatched_pass,unmatched_presto=assemble('primer_unmatched',unmatched_pair)
combined=MERGED/f'{RUN}_all_assemble-pass.fastq.gz'
with gzip.open(combined,'wb',compresslevel=1) as out:
 for src in (primer_pass,unmatched_pass):
  with gzip.open(src,'rb') as inp:shutil.copyfileobj(inp,out)
assembled=count_fastq_records(combined);failed=primer_presto['fail']+unmatched_presto['fail'];total=primer_presto['pairs']+unmatched_presto['pairs']
assert assembled==primer_presto['pass']+unmatched_presto['pass']
summary={'input_pairs':total,'assembled_pairs':assembled,'failed_pairs':failed,'merge_rate':assembled/total,'primer_pass_assembled':primer_presto['pass'],'primer_unmatched_assembled':unmatched_presto['pass']}
(MERGED/'assembly_summary.json').write_text(json.dumps(summary,indent=2)+chr(10));print(summary)


In [ ]:
# Exact collapse и одинаковая IgBLAST-аннотация лёгких цепей с базой BALB/cByJ.
WORK=TARGET/'annotation';WORK.mkdir(parents=True,exist_ok=True);fasta=WORK/f'{RUN}_all_assemble-pass.fasta';collapsed=WORK/f'{RUN}.collapse-unique.fasta'
with gzip.open(combined,'rt') as ih,fasta.open('w') as oh:
 while True:
  head=ih.readline()
  if not head:break
  seq=ih.readline().strip();ih.readline();ih.readline();oh.write('>'+head[1:].split()[0]+chr(10)+seq+chr(10))
run([ENV/'bin/CollapseSeq.py','-s',fasta,'-o',collapsed,'--fasta','-n','0'],WORK/'collapse.stdout.log',WORK/'collapse.stderr.log')
airr=WORK/f'{RUN}_balbcbyj_igkl.airr.tsv';tmp=Path(str(airr)+'.rerun.tmp');tmp.unlink(missing_ok=True);expected=fasta_count(collapsed);env=os.environ.copy();env['IGDATA']=str(GERMLINE/'igdata_balbcbyj')
cmd=[ENV/'bin/igblastn','-query',collapsed,'-organism','balbcbyj','-ig_seqtype','Ig','-germline_db_V',GERMLINE/'igblast_db/balbcbyj_igkv_iglv','-germline_db_D',GERMLINE/'ncbi_mouse/mouse_gl_D','-germline_db_J',GERMLINE/'igblast_db/all_strains_IGKLJ','-auxiliary_data',GERMLINE/'ogrdb_balbcbyj/all_strains_IGKLJ.aux','-domain_system','imgt','-outfmt','19','-num_threads','8','-out',tmp]
run(cmd,WORK/'igblast.stdout.log',WORK/'igblast.stderr.log',env=env);assert tsv_count(tmp)==expected,(tsv_count(tmp),expected);assert_airr_query_identity(collapsed,tmp);tmp.replace(airr)
verified=assert_airr_query_identity(collapsed,airr);print('AIRR rows/query IDs verified',verified)
truth=build_airr_truth(input_tsv=airr,output_dir=WORK/'strict_truth',run=f'{RUN}_{TARGET_BRANCH}',source=f'PRJNA1226555_{TARGET_BRANCH}',expected_loci={'IGK','IGL'})
print(truth)


In [ ]:
# Сравнение биологического выхода и целостности V-региона с выбранной базовой веткой paper4.
def yes(v):return str(v).lower() in {'true','t','1','yes'}
def summarize(path,strict_summary):
 raw=Counter();loci=Counter();vstarts=[];valens=[];vcalls=Counter();ck='GATGGTGGGAAGATGGATAC'
 with open(path,errors='replace') as h:
  for r in csv.DictReader(h,delimiter='\t'):
   raw['rows']+=1;raw['productive']+=yes(r.get('productive'));raw['complete_vdj']+=yes(r.get('complete_vdj'));raw['productive_complete_no_stop']+=yes(r.get('productive')) and yes(r.get('complete_vdj')) and not yes(r.get('stop_codon'))
   loci[r.get('locus') or 'UNASSIGNED']+=1;vcalls[r.get('v_call') or 'UNASSIGNED']+=1
   try:vstarts.append(int(r['v_sequence_start']))
   except (ValueError,KeyError):pass
   va=(r.get('v_sequence_alignment') or '').replace('-','').replace('.','')
   if va:valens.append(len(va))
   seq=(r.get('sequence') or '').upper();raw['ck_full_exact']+=ck in seq
 return {'raw':dict(raw),'loci':dict(loci),'strict':strict_summary,'v_sequence_start_median':statistics.median(vstarts) if vstarts else None,'v_alignment_length_median':statistics.median(valens) if valens else None,'top_v_calls':vcalls.most_common(20)}
source_airr=SOURCE/'annotation'/f'{RUN}_balbcbyj_igkl.airr.tsv';source_summary=report[selected_label]['strict']
comparison={'source_branch':SOURCE_BRANCH,'candidate_branch':TARGET_BRANCH,'ck_source_status':'empirical_candidate_not_kit_source_backed','paper4_cut':summarize(source_airr,source_summary),'paper4_plus_ck_cut':summarize(airr,truth)}
comparison['candidate_passes_noninferiority']=comparison['paper4_plus_ck_cut']['strict']['accepted_source_records']>=0.99*comparison['paper4_cut']['strict']['accepted_source_records']
def promote_staged_candidate(staged,final):
 backup=Path(str(final)+'.previous');shutil.rmtree(backup,ignore_errors=True)
 if final.exists():final.replace(backup)
 try:staged.replace(final)
 except Exception:
  if backup.exists() and not final.exists():backup.replace(final)
  raise
 shutil.rmtree(backup,ignore_errors=True)
promote_staged_candidate(TARGET,TARGET_FINAL)
comparison_out=BASE/'comparisons/light_primer_strategy_comparison.json';comparison_tmp=Path(str(comparison_out)+'.tmp');comparison_tmp.write_text(json.dumps(comparison,indent=2)+chr(10));comparison_tmp.replace(comparison_out);print(json.dumps(comparison,indent=2));print(comparison_out)
